# Módulo 0 — Carga de datos

| | |
|---|---|
| **Responde** | ítems 1 y 2 |
| **Código** | `carga_de_datos.py` (en esta misma carpeta) |
| **Recibe** | archivos de `datos/registros_crudos/` y `datos/lista_de_registros.csv` |
| **Devuelve** | un `Registro` por archivo, con `ecg` y `fs` (Hz) |
| **Responsable** | _completar_ |

> Las funciones se programan en `carga_de_datos.py`. Este cuaderno sólo las llama, grafica y discute.
> Para guardar una figura para el informe: `graficos.guardar_figura(fig, "nombre")`.
> Para comprobar el módulo: correr `python main.py` desde la carpeta del repo.

In [1]:
# Permite usar el código de las otras carpetas del repo
import sys
sys.path.insert(0, "../..")

# Recarga los cambios que hagas en los archivos .py sin reiniciar el cuaderno
%load_ext autoreload
%autoreload 2

import numpy as np
import matplotlib.pyplot as plt

from modulos.m0_carga_de_datos import carga_de_datos
from modulos.herramientas import ecg_de_prueba, graficos
from modulos.herramientas.procesamiento_completo import procesar, procesar_todos

graficos.aplicar_estilo()

## Lista de registros

In [2]:
carga_de_datos.leer_lista_de_registros()

,archivo,voluntario,condicion,fecha,notas
0,ECG_data_pre.csv,V1,reposo,,
1,ECG_data_post.csv,V1,post_ejercicio,,


## Ítem 1 — Formato del registro

La interfaz guarda un **CSV de una sola columna**: encabezado `Sample` y una muestra por fila. Cada valor es la tensión a la salida del AD8232 en voltios ($V = 5n/1023$, redondeado a dos decimales, es decir, a pasos de 10 mV). No hay columna de tiempo, ni frecuencia de muestreo, ni unidades, ni datos del sujeto. `cargar_registro()` lee esa columna y le asigna `fs` y las unidades fijas de `carga_de_datos.py`.

Los otros formatos de ECG (SCP-ECG, HL7 aECG, DICOM, WFDB, EDF…) están en la Introducción del informe (sección 1.2).

In [3]:
# --- Inspección del archivo crudo ---
ruta = carga_de_datos.CARPETA_REGISTROS_CRUDOS / "ECG_data_pre.csv"
with open(ruta, encoding="utf-8") as archivo:
    primeras = [next(archivo).rstrip() for _ in range(6)]
print("Primeras líneas:", primeras)

reg = carga_de_datos.cargar_registro(ruta)
niveles = np.unique(reg.ecg)
print(f"{len(reg.ecg)} muestras, de {reg.ecg.min():.2f} a {reg.ecg.max():.2f} {reg.unidades}")
print(f"Paso entre niveles distintos: {np.min(np.diff(niveles)):.3f} V (redondeo a dos decimales)")

Primeras líneas: ['Sample', '1.52', '1.52', '1.52', '1.52', '1.52']
28800 muestras, de 1.28 a 3.03 V
Paso entre niveles distintos: 0.010 V (redondeo a dos decimales)


## Ítem 2 — Frecuencia de muestreo

El archivo no la guarda, así que se deduce del firmware y se contrasta con los registros:

- El firmware configura el Timer1 en modo CTC: $f_s = 16\,\text{MHz} / (64 \cdot (1 + \texttt{OCR1A}))$. Con `OCR1A = 499` da **500 Hz** (con 999, la otra opción, daría 250 Hz).
- **Duración**: a 500 Hz los registros duran 57,6 s y 173,2 s, lo que pide el protocolo (1 y 3 min). A 250 Hz durarían el doble.
- **Paquetes**: el Arduino envía paquetes de 100 muestras y entre paquetes se pierden algunas muestras. Si la señal está agrupada en bloques de 100, el salto entre muestras consecutivas tiene que ser mayor justo en el borde de cada bloque, y no en bloques de otro tamaño.

In [4]:
# --- Evidencia de la frecuencia de muestreo ---
for reg in carga_de_datos.cargar_todos():
    print(f"{reg.nombre:20s} {len(reg.ecg):6d} muestras:  {len(reg.ecg)/250:6.1f} s a 250 Hz   "
          f"{len(reg.ecg)/500:6.1f} s a 500 Hz")

# Salto medio entre muestras consecutivas según la posición dentro de un bloque de B muestras.
# Para cada B se busca la posición con el salto más grande y se lo compara con el resto.
print("\nSalto en la posición más saltona del bloque / salto medio en las demás posiciones")
for reg in carga_de_datos.cargar_todos():
    salto = np.abs(np.diff(reg.ecg))[500:]          # se saltea el primer segundo (paquetes viejos)
    pos = np.arange(1, len(reg.ecg))[500:]
    cocientes = {}
    for B in (80, 90, 100, 110, 128):
        medio = np.array([salto[pos % B == k].mean() for k in range(B)])
        k = medio.argmax()
        cocientes[B] = medio[k] / np.delete(medio, k).mean()
    print(f"  {reg.nombre:20s}", "   ".join(f"B={B}: {c:.1f}" for B, c in cocientes.items()))

V1_reposo             28800 muestras:   115.2 s a 250 Hz     57.6 s a 500 Hz
V1_post_ejercicio     86600 muestras:   346.4 s a 250 Hz    173.2 s a 500 Hz

Salto en la posición más saltona del bloque / salto medio en las demás posiciones


  V1_reposo            B=80: 1.9   B=90: 1.8   B=100: 4.2   B=110: 1.9   B=128: 1.4


  V1_post_ejercicio    B=80: 1.8   B=90: 1.5   B=100: 4.6   B=110: 1.5   B=128: 1.3


## Los registros cargados

In [5]:
registros = carga_de_datos.cargar_todos()
for reg in registros:
    print(f"{reg.nombre:20s} fs = {reg.fs:g} Hz   duración = {reg.duracion:.1f} s   unidades: {reg.unidades}")

V1_reposo            fs = 500 Hz   duración = 57.6 s   unidades: V
V1_post_ejercicio    fs = 500 Hz   duración = 173.2 s   unidades: V


Sólo en bloques de 100 muestras aparece un salto mucho mayor (unas 4 veces) en una posición fija: son los bordes de los paquetes. Con los 500 Hz del timer y unas 4 muestras perdidas por paquete, la tasa media queda en ~480 muestras/s, que es lo que permite el puerto serie a 9600 baudios. En el análisis se usa **fs = 500 Hz**, la separación real entre muestras dentro de cada paquete (sección 2.2.3 del informe).